# Advanced Retail Analytics (Product)

## SECTION 1: Data Loading & Quality Audits

In [2]:
import pandas as pd

# Load the Global Superstore dataset, specifying 'latin1' encoding
df = pd.read_csv('/content/Global_Superstore2.csv', encoding='latin1')

# Preview first 5 rows
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,32298,CA-2012-124891,31-07-2012,31-07-2012,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,05-02-2013,07-02-2013,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,17-10-2013,18-10-2013,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,28-01-2013,30-01-2013,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,05-11-2013,06-11-2013,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical


In [3]:
# Basic dataset structure
print("Rows, Columns:", df.shape)

# Detailed column information
df.info()


Rows, Columns: (51290, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          51290 non-null  int64  
 1   Order ID        51290 non-null  object 
 2   Order Date      51290 non-null  object 
 3   Ship Date       51290 non-null  object 
 4   Ship Mode       51290 non-null  object 
 5   Customer ID     51290 non-null  object 
 6   Customer Name   51290 non-null  object 
 7   Segment         51290 non-null  object 
 8   City            51290 non-null  object 
 9   State           51290 non-null  object 
 10  Country         51290 non-null  object 
 11  Postal Code     9994 non-null   float64
 12  Market          51290 non-null  object 
 13  Region          51290 non-null  object 
 14  Product ID      51290 non-null  object 
 15  Category        51290 non-null  object 
 16  Sub-Category    51290 non-null  object 
 17  Prod

In [4]:
# Convert date columns to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], errors='coerce')

# Verify conversion worked
df[['Order Date', 'Ship Date']].head()


/tmp/ipython-input-1017373805.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
/tmp/ipython-input-1017373805.py:3: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Ship Date'] = pd.to_datetime(df['Ship Date'], errors='coerce')


,Order Date,Ship Date
0,2012-07-31,2012-07-31
1,2013-02-05,2013-02-07
2,2013-10-17,2013-10-18
3,2013-01-28,2013-01-30
4,2013-11-05,2013-11-06


In [5]:
# Check for impossible values

print("Zero or negative sales:", (df['Sales'] <= 0).sum())
print("Zero or negative quantity:", (df['Quantity'] <= 0).sum())

# Profit distribution (to spot extreme outliers)
df['Profit'].describe()

Zero or negative sales: 0
Zero or negative quantity: 0


,Profit
count,51290.000000
mean,28.610982
std,174.340972
min,-6599.978000
25%,0.000000
50%,9.240000
75%,36.810000
max,8399.976000


## SECTION 2 — PRODUCT FEATURE ENGINEERING (CORE ANALYTICS LAYER)

In [6]:
# ---- Product analytics feature engineering ---- #

# Unit price per item
df['Unit_Price'] = df['Sales'] / df['Quantity']

# Profit margin (handle division safely)
df['Profit_Margin'] = df['Profit'] / df['Sales']

# Weekly reporting features
df['Order_Year'] = df['Order Date'].dt.year
df['Order_Week'] = df['Order Date'].dt.isocalendar().week

# Shipping lag (days between order and ship)
df['Shipping_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Preview new features
df[['Sales','Quantity','Unit_Price','Profit','Profit_Margin','Order_Year','Order_Week','Shipping_Days']].head()

,Sales,Quantity,Unit_Price,Profit,Profit_Margin,Order_Year,Order_Week,Shipping_Days
0,2309.650,7,329.950,762.1845,0.330000,2012,31,0
1,3709.395,9,412.155,-288.7650,-0.077847,2013,6,2
2,5175.171,9,575.019,919.9710,0.177766,2013,42,1
3,2892.510,5,578.502,-96.5400,-0.033376,2013,5,2
4,2832.960,8,354.120,311.5200,0.109963,2013,45,1


I grouped the raw transaction data at the product (SKU) level to create a performance table where each product now has total sales, total profit, average profit margin, total units sold, average unit price, and number of orders. For example, instead of seeing hundreds of rows for one chair product, we now see one row showing how much that chair sold overall, how profitable it was, and how frequently customers bought it. This transforms raw sales logs into a true product analytics dataset that can be used to identify top-performing products, loss-making SKUs, pricing issues, and growth opportunities.

## Section 3: Product Performance Deep Dive

In [9]:
# ============================================================
# STEP 1: Build Product-Level Performance Table
# ------------------------------------------------------------
# Objective:
# Convert transaction-level data into SKU-level analytics.
# Each product will now have:
#   - Total Sales
#   - Total Profit
#   - Average Profit Margin
#   - Total Quantity Sold
#   - Average Unit Price
#   - Number of Orders
#
# Why this matters:
# Instead of analyzing 51,000+ transactions,
# we now evaluate each product as a business unit.
# This is the foundation for identifying winners,
# loss-makers, and pricing issues.
# ============================================================

product_perf = (
    df.groupby(['Product ID', 'Product Name', 'Category', 'Sub-Category'])
      .agg(
          Total_Sales=('Sales', 'sum'),
          Total_Profit=('Profit', 'sum'),
          Avg_Margin=('Profit_Margin', 'mean'),
          Total_Quantity=('Quantity', 'sum'),
          Avg_Unit_Price=('Unit_Price', 'mean'),
          Orders_Count=('Order ID', 'count')
      )
      .reset_index()
)

# Preview the aggregated SKU-level table
product_perf.head()


,Product ID,Product Name,Category,Sub-Category,Total_Sales,Total_Profit,Avg_Margin,Total_Quantity,Avg_Unit_Price,Orders_Count
0,FUR-ADV-10000002,"Advantus Photo Frame, Duo Pack",Furniture,Furnishings,159.120,60.390,0.379525,3,53.04000,2
1,FUR-ADV-10000108,"Advantus Clock, Erganomic",Furniture,Furnishings,350.070,3.360,0.009598,7,50.01000,3
2,FUR-ADV-10000183,"Advantus Photo Frame, Black",Furniture,Furnishings,974.832,-651.738,-0.650623,31,40.39725,8
3,FUR-ADV-10000188,"Advantus Stacking Tray, Erganomic",Furniture,Furnishings,124.950,4.200,-0.219488,7,18.49260,5
4,FUR-ADV-10000190,"Advantus Frame, Duo Pack",Furniture,Furnishings,222.360,104.460,0.469779,2,111.18000,1


In [10]:
# ============================================================
# STEP 2: Identify Top Revenue Drivers vs Top Profit Drivers
# ------------------------------------------------------------
# Objective:
# Rank products to understand:
#   - Which SKUs generate the most sales volume (revenue leaders)
#   - Which SKUs generate the most profit (true business value)
#
# Why this matters:
# High sales does NOT always mean high profitability.
# Some products sell a lot but lose margin due to discounts and costs.
# This step separates revenue growth from profit growth.
# ============================================================

# Top 10 products by total sales
top_sales_products = product_perf.sort_values(
    by='Total_Sales', ascending=False
).head(10)

# Top 10 products by total profit
top_profit_products = product_perf.sort_values(
    by='Total_Profit', ascending=False
).head(10)

# Display key metrics for comparison
top_sales_products[['Product Name','Total_Sales','Total_Profit','Avg_Margin']]

,Product Name,Total_Sales,Total_Profit,Avg_Margin
9388,Canon imageCLASS 2200 Advanced Copier,61599.8240,2.519993e+04,0.384667
10632,"Nokia Smart Phone, with Caller ID",30041.5482,5.455948e+03,0.188564
4017,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.3840,7.753039e+03,0.050000
9721,Cisco TelePresence System EX90 Videoconferenci...,22638.4800,-1.811078e+03,-0.080000
10646,"Nokia Smart Phone, Full Size",22262.1000,8.121480e+03,0.367976
745,HON 5400 Series Task Chairs for Big and Tall,21870.5760,8.526513e-14,-0.014683
541,"SAFCO Executive Leather Armchair, Black",21329.7300,1.363230e+03,0.019981
2882,"Hoover Stove, Red",21147.0840,1.034558e+04,0.476163
3744,GBC DocuBind TL300 Electric Binding System,19823.4790,2.233505e+03,-0.084091
3634,GBC Ibimaster 500 Manual ProClick Binding System,19024.5000,7.609800e+02,-0.398148


From the top-sales product list, I can clearly see that products like the Canon imageCLASS 2200 Advanced Copier and Nokia Smart Phones are both generating very high sales and strong profits. These products are the main revenue drivers of the business and also contribute positively to profitability, which means the company should continue investing in and promoting them.

At the same time, some products such as the Cisco TelePresence System and certain binding machines show high sales but negative profit margins. This means they are selling in large volumes but are actually causing financial losses, most likely due to heavy discounts, high shipping costs, or pricing that is too low.

This comparison shows that high sales alone do not guarantee profitability. It highlights the importance of analyzing both revenue and profit together to identify strong products as well as pricing or cost issues.

In [11]:
# ============================================================
# STEP 3: Identify Loss-Making and Margin-Erosion Products
# ------------------------------------------------------------
# Objective:
# Find products that are actively hurting profitability.
# These may sell well but generate negative or very low margins.
#
# Why this matters:
# These SKUs are prime candidates for:
#   - Price correction
#   - Discount strategy review
#   - Removal from assortment
# ============================================================

# Products with negative total profit
loss_making_products = product_perf[
    product_perf['Total_Profit'] < 0
].sort_values('Total_Profit')

# Display worst 10 loss-making products
loss_making_products.head(10)[
    ['Product Name','Total_Sales','Total_Profit','Avg_Margin']
]

,Product Name,Total_Sales,Total_Profit,Avg_Margin
9574,Cubify CubeX 3D Printer Double Head Print,11099.9630,-8879.9704,-0.952778
2614,"Hoover Stove, White",11728.8270,-4958.1630,-0.301877
9605,Lexmark MX611dhe Monochrome Laser Printer,16829.9010,-4589.9730,-0.361111
10424,"Apple Smart Phone, Full Size",7259.1561,-4574.6439,-0.520158
9984,"Motorola Smart Phone, Cordless",10348.7580,-3998.6820,-0.439345
9876,Cubify CubeX 3D Printer Triple Head Print,7999.9800,-3839.9904,-0.480000
699,"Office Star Executive Leather Armchair, Black",6497.2770,-3066.7830,-0.457067
1993,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.6400,-2876.1156,-0.247000
2157,"Lesro Computer Table, Fully Assembled",1199.3520,-2798.4880,-2.333333
2118,"Hon Conference Table, Rectangular",3667.0095,-2619.3105,-0.688495


From this table, I can see that several products are generating significant negative profits despite having decent sales.

For example:

Cubify CubeX 3D Printer (Double Head Print) generated around $11,099 in sales but resulted in a loss of nearly $8,880, with an extremely negative average margin of -95%. This means the company is almost losing money on every sale of this product.

Hoover Stove (White) and Lexmark Monochrome Laser Printer also show strong sales but heavy losses, indicating that pricing or discount strategy is not sustainable.

Some furniture items like Hon Conference Table and Office Star Executive Armchair also show consistent negative margins, which suggests cost structure or discounting issues.

This clearly shows that certain products are damaging overall profitability. Even though they bring revenue, they reduce net earnings. These products should be reviewed for price correction, cost reduction, or possible removal from the assortment.

In [12]:
# ============================================================
# STEP 4: Analyze Impact of Discount on Profit Margin
# ------------------------------------------------------------
# Objective:
# Understand how increasing discount levels affect profitability.
#
# Why this matters:
# High discounts often drive sales volume but destroy margins.
# This analysis helps identify:
#   - Whether discounts are sustainable
#   - The point where profit starts turning negative
# ============================================================

# Create discount buckets for easier analysis
df['Discount_Bucket'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1],
    labels=['0%', '0-10%', '10-20%', '20-30%', '30-50%', '50%+']
)

# Calculate average margin and sales per discount bucket
discount_analysis = (
    df.groupby('Discount_Bucket')
      .agg(
          Avg_Margin=('Profit_Margin', 'mean'),
          Avg_Sales=('Sales', 'mean'),
          Orders=('Order ID', 'count')
      )
      .reset_index()
)

discount_analysis

/tmp/ipython-input-1398460475.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('Discount_Bucket')


,Discount_Bucket,Avg_Margin,Avg_Sales,Orders
0,0%,0.264898,241.042813,29009
1,0-10%,0.171293,419.452628,4679
2,10-20%,0.137450,280.086282,6274
3,20-30%,-0.038511,395.609813,967
4,30-50%,-0.337934,190.019620,6189
5,50%+,-1.147385,89.075902,4172


From this table, I can clearly see that as discount levels increase, the average profit margin consistently decreases.

When there is no discount, the company earns a strong average margin of around 26%, which is very healthy.
With small discounts of 0–10% and 10–20%, the margin is still positive but already drops to around 13–17%.

However, once discounts reach 20–30%, the average margin becomes negative, meaning the company starts losing money on each sale.
The situation becomes much worse at 30–50% and above 50% discounts, where margins fall sharply and reach extremely high losses.

Even though discounted products sometimes show higher average sales value, the profitability collapses.

This clearly shows that aggressive discounting is the main driver of loss-making products and is not a sustainable pricing strategy.

In [13]:
# ============================================================
# STEP 5: Weekly Product Performance Report
# ------------------------------------------------------------
# Objective:
# Track how sales and profit change week by week.
#
# Why this matters:
# Businesses monitor weekly performance to:
#   - Spot growth trends
#   - Detect sudden drops or spikes
#   - Understand seasonality and demand patterns
# ============================================================

# Aggregate weekly performance across all products
weekly_performance = (
    df.groupby(['Order_Year', 'Order_Week'])
      .agg(
          Weekly_Sales=('Sales', 'sum'),
          Weekly_Profit=('Profit', 'sum'),
          Weekly_Orders=('Order ID', 'count'),
          Avg_Weekly_Margin=('Profit_Margin', 'mean')
      )
      .reset_index()
      .sort_values(['Order_Year', 'Order_Week'])
)

# Preview weekly report
weekly_performance.head(10)

,Order_Year,Order_Week,Weekly_Sales,Weekly_Profit,Weekly_Orders,Avg_Weekly_Margin
0,2011,1,25827.28344,2752.07684,93,0.035182
1,2011,2,27169.67248,2862.55758,113,-0.083614
2,2011,3,21799.13634,3536.63394,96,0.094097
3,2011,4,15337.02860,-2220.80560,104,-0.114997
4,2011,5,20268.44130,2290.55620,90,0.058241
5,2011,6,23255.99352,4826.99292,98,0.066576
6,2011,7,27372.64350,4406.60490,93,0.084099
7,2011,8,25479.34666,2799.65236,107,-0.001241
8,2011,9,30824.71170,1742.60830,126,0.048362
9,2011,10,26137.75572,5506.15722,97,0.105311


From this weekly table, I can see how the business performs on a week-by-week basis instead of only yearly or monthly trends.

Some weeks, such as Week 6 and Week 10 of 2011, show very strong profits, which means product mix and pricing worked well during those periods.
However, other weeks like Week 4 and Week 8 show negative or almost zero profit margins even though sales were still high.

This indicates that during certain weeks, heavy discounting or high-cost orders reduced profitability even when demand was strong.

Overall, this weekly view helps identify:

High-performing weeks where pricing strategy worked well

Risk weeks where margins collapsed

Short-term trends that yearly dashboards cannot show

This is very useful for planning promotions, inventory, and pricing decisions.

## SECTION 4 – TPD & ACV ANALYTICS

In [24]:
# ============================================================
# STEP 4.1: Total Points of Distribution (TPD Proxy)
# ------------------------------------------------------------
# Objective:
# Measure how widely each product is sold across markets/regions.
#
# Why this matters:
# Products with wider distribution usually have higher visibility.
# Low distribution + high margin = growth opportunity.
# High distribution + low margin = pricing problem.
# ============================================================

# Count how many unique regions each product appears in
product_distribution = (
    df.groupby(['Product ID', 'Product Name'])
      .agg(
          Distribution_Points=('Region', 'nunique'),
          Total_Sales=('Sales', 'sum'),
          Total_Profit=('Profit', 'sum')
      )
      .reset_index()
)

product_distribution.head()


,Product ID,Product Name,Distribution_Points,Total_Sales,Total_Profit
0,FUR-ADV-10000002,"Advantus Photo Frame, Duo Pack",2,159.120,60.390
1,FUR-ADV-10000108,"Advantus Clock, Erganomic",1,350.070,3.360
2,FUR-ADV-10000183,"Advantus Photo Frame, Black",2,974.832,-651.738
3,FUR-ADV-10000188,"Advantus Stacking Tray, Erganomic",3,124.950,4.200
4,FUR-ADV-10000190,"Advantus Frame, Duo Pack",1,222.360,104.460


Here, Distribution_Points shows how many different regions each product is sold in.

For example:

Advantus Stacking Tray, Ergonomic is sold in 3 regions, which means it has wider market reach.

Advantus Clock, Ergonomic is sold in only 1 region, meaning its availability is limited.

When we compare this with sales and profit:

Some products with wider distribution still generate low or negative profit (like the Advantus Photo Frame, Black), which suggests that simply expanding reach does not guarantee profitability.

Other products with limited distribution but good profit (like Advantus Frame, Duo Pack) may be strong candidates for expansion into more regions.

This helps answer important business questions such as:

• Which products should be expanded into more markets?
• Which widely distributed products need pricing or cost correction

In [25]:
# ============================================================
# STEP 4.2: Top & Low Products by Distribution Points (TPD Ranking)
# ------------------------------------------------------------
# Objective:
# Identify products with the widest physical/market presence.
#
# Why this matters:
# High distribution = high visibility + demand opportunity.
# These are the products customers see most often.
# ============================================================

top_tpd_products = product_distribution.sort_values(
    by='Distribution_Points', ascending=False
).head(10)

top_tpd_products[['Product Name','Distribution_Points','Total_Sales','Total_Profit']]


,Product Name,Distribution_Points,Total_Sales,Total_Profit
8454,"Logitech Memory Card, Erganomic",4,2332.4295,-331.2105
8453,"Enermax Flash Drive, Bluetooth",4,858.0800,-1.4400
8451,SanDisk Cruzer 64 GB USB Flash Drive,4,1184.0320,319.6160
7293,"Tenex File Cart, Single Width",4,3892.9212,834.3612
1554,"Tenex Light Bulb, Durable",4,222.3760,-87.8240
1552,Coloredge Poster Frame,4,298.2000,116.2980
1582,"Advantus Light Bulb, Black",4,409.4280,33.4680
1581,"Eldon Light Bulb, Durable",4,517.5144,199.2744
5349,"OIC Staples, Bulk Pack",4,330.7878,-47.2122
5350,"Stockwell Rubber Bands, 12 Pack",4,220.7826,30.9726


In [26]:
# Products with very limited reach (potential growth or poor demand)

low_tpd_products = product_distribution.sort_values(
    by='Distribution_Points'
).head(10)

low_tpd_products[['Product Name','Distribution_Points','Total_Sales','Total_Profit']]


,Product Name,Distribution_Points,Total_Sales,Total_Profit
19,"Advantus Clock, Black",1,66.807,-6.213
2559,"Hoover Stove, Black",1,8291.340,-68.340
2555,"Cuisinart Microwave, White",1,737.600,228.640
2552,"Hoover Toaster, Silver",1,101.268,-18.612
2548,"Cuisinart Toaster, Red",1,64.152,-32.088
2546,"KitchenAid Stove, Black",1,2050.488,-512.712
2543,"Hamilton Beach Stove, Black",1,1950.372,-1121.688
2542,"KitchenAid Microwave, Black",1,371.700,43.320
2536,"Breville Toaster, Black",1,101.520,-149.780
2533,"Hoover Blender, Black",1,96.600,-61.860


📊 Interpretation – Top Products by Distribution Reach (TPD)

From the top TPD list, I can see that products like:

Logitech Memory Card, Ergonomic

SanDisk Cruzer USB Flash Drive

Tenex File Cart

Light bulbs and office supplies

are sold in all four regions, meaning they have the widest market reach.

However, not all of these widely distributed products are profitable.
For example:

Logitech Memory Card and Enermax Flash Drive have wide reach but negative profit.

Tenex File Cart and SanDisk USB Drive combine wide reach with strong profit.

This shows that wide distribution alone does not guarantee success — pricing and cost control still matter.

📉 Interpretation – Low Distribution Products (Limited Reach)

From the low TPD list, I can see that many products are sold in only one region, such as:

Cuisinart Microwave

KitchenAid appliances

Hamilton Beach Stove

Some of these limited-reach products still generate good profit (like the Cuisinart Microwave, White), which suggests strong demand where available and potential for expansion into more regions.

Others show both low reach and negative profit, indicating weak products that may not be worth scaling.

In [27]:
# ============================================================
# STEP 4.3: ACV-Style Weighted Distribution (Sales-Weighted Reach)
# ------------------------------------------------------------
# Objective:
# Weight product distribution by the sales importance of regions.
#
# Why this matters:
# Being present in high-revenue regions is more valuable
# than being present in low-revenue regions.
# ============================================================

# Total sales per region (market size proxy)
region_sales = df.groupby('Region')['Sales'].sum().reset_index()
region_sales.columns = ['Region', 'Region_Total_Sales']

# Merge region weights back to transaction data
df_weighted = df.merge(region_sales, on='Region', how='left')

# Compute ACV-style metric per product
product_acv = (
    df_weighted.groupby(['Product ID', 'Product Name'])
      .agg(
          ACV_Weighted_Distribution=('Region_Total_Sales', 'sum'),
          Total_Sales=('Sales', 'sum'),
          Total_Profit=('Profit', 'sum')
      )
      .reset_index()
)

product_acv.head()


,Product ID,Product Name,ACV_Weighted_Distribution,Total_Sales,Total_Profit
0,FUR-ADV-10000002,"Advantus Photo Frame, Duo Pack",1589934.522,159.120,60.390
1,FUR-ADV-10000108,"Advantus Clock, Erganomic",2351319.633,350.070,3.360
2,FUR-ADV-10000183,"Advantus Photo Frame, Black",6404514.288,974.832,-651.738
3,FUR-ADV-10000188,"Advantus Stacking Tray, Erganomic",3269185.314,124.950,4.200
4,FUR-ADV-10000190,"Advantus Frame, Duo Pack",806161.311,222.360,104.460


In [28]:
# ============================================================
# STEP 4.4: Top Products by ACV-Weighted Distribution
# ------------------------------------------------------------
# Objective:
# Identify products with the strongest presence in high-sales regions.
# ============================================================

top_acv_products = product_acv.sort_values(
    by='ACV_Weighted_Distribution', ascending=False
).head(10)

top_acv_products[['Product Name','ACV_Weighted_Distribution','Total_Sales','Total_Profit']]


,Product Name,ACV_Weighted_Distribution,Total_Sales,Total_Profit
3376,"Sanford Pencil Sharpener, Easy-Erase",4.925408e+07,2289.105,1028.025
3494,"BIC Canvas, Fluorescent",4.898165e+07,3946.128,1194.918
2933,"Binney & Smith Sketch Pad, Blue",4.846573e+07,3900.144,574.014
3284,"Boston Sketch Pad, Blue",4.683806e+07,3465.180,1413.900
3467,"Boston Canvas, Fluorescent",4.526392e+07,4309.650,437.070
7283,"Fellowes Box, Industrial",4.523715e+07,1368.864,-145.566
3170,"Binney & Smith Sketch Pad, Water Color",4.491118e+07,3346.200,924.060
2930,"BIC Highlighters, Water Color",4.491118e+07,1397.352,235.752
3607,"Ibico Binding Machine, Recycled",4.401576e+07,3546.930,360.930
3004,"Binney & Smith Highlighters, Water Color",4.368978e+07,1330.521,526.221


From this ranked ACV table, I can see that products like:

Sanford Pencil Sharpener, Easy-Erase

BIC Canvas, Fluorescent

Binney & Smith Sketch Pads

have the strongest presence in the highest-sales regions.

This means these products are not just selling — they are selling in the most valuable markets where most revenue is generated. These are strategically important products for the business because they dominate high-value regions.

Most of these products also show positive profits, which indicates a strong combination of:

✔ Wide high-value distribution
✔ Healthy profitability

These products should be prioritized for stock availability, promotions, and expansion.

⚠️ Important secondary insight

I can also see a product like Fellowes Box, Industrial appearing high in ACV but with negative profit, which means:

It is widely present in major markets

But it is losing money

This is a classic pricing or cost issue that needs immediate attention.

## SECTION 5: Assortment Optimization

In [30]:
# ============================================================
# STEP 5.1: Assortment Optimization & Product Strategy Scoring
# ------------------------------------------------------------
# Objective:
# Combine performance + pricing + distribution into clear decisions:
#   - Keep (strong performers)
#   - Expand (high profit but low reach)
#   - Fix Pricing (high reach but low profit)
#   - Drop (low reach + low profit)
# ============================================================

# Merge performance and distribution data
product_strategy = product_perf.merge(
    product_distribution[['Product ID','Distribution_Points']],
    on='Product ID',
    how='left'
)

# Basic classification logic
def classify_product(row):
    if row['Total_Profit'] > 500 and row['Distribution_Points'] >= 3:
        return 'Core Product (Keep & Promote)'
    elif row['Total_Profit'] > 500 and row['Distribution_Points'] < 3:
        return 'Growth Opportunity (Expand Reach)'
    elif row['Total_Profit'] <= 0 and row['Distribution_Points'] >= 3:
        return 'Pricing Issue (Fix Margin)'
    else:
        return 'Weak Product (Review/Drop)'

product_strategy['Product_Action'] = product_strategy.apply(classify_product, axis=1)

product_strategy[['Product Name','Total_Sales','Total_Profit','Distribution_Points','Product_Action']].head(10)


,Product Name,Total_Sales,Total_Profit,Distribution_Points,Product_Action
0,"Advantus Photo Frame, Duo Pack",159.120,60.390,2,Weak Product (Review/Drop)
1,"Advantus Clock, Erganomic",350.070,3.360,1,Weak Product (Review/Drop)
2,"Advantus Photo Frame, Black",974.832,-651.738,2,Weak Product (Review/Drop)
3,"Advantus Stacking Tray, Erganomic",124.950,4.200,3,Weak Product (Review/Drop)
4,"Advantus Frame, Duo Pack",222.360,104.460,1,Weak Product (Review/Drop)
5,"Advantus Frame, Erganomic",2194.800,702.000,2,Growth Opportunity (Expand Reach)
6,"Advantus Clock, Duo Pack",205.800,24.600,1,Weak Product (Review/Drop)
7,"Advantus Stacking Tray, Black",290.730,81.180,2,Weak Product (Review/Drop)
8,"Advantus Frame, Black",177.792,-88.968,1,Weak Product (Review/Drop)
9,"Advantus Door Stop, Black",959.139,176.379,2,Weak Product (Review/Drop)


In [32]:
# ============================================================
# STEP 5.2: Top Priority Products in Each Strategy Group
# ------------------------------------------------------------
# Objective:
# Identify the most important products to act on in each category.
# ============================================================

# Core products to keep and promote
top_keep = product_strategy[
    product_strategy['Product_Action'] == 'Core Product (Keep & Promote)'
].sort_values('Total_Profit', ascending=False).head(5)

# Products to expand into more regions
top_expand = product_strategy[
    product_strategy['Product_Action'] == 'Growth Opportunity (Expand Reach)'
].sort_values('Total_Profit', ascending=False).head(5)

# Products needing pricing correction
top_fix_pricing = product_strategy[
    product_strategy['Product_Action'] == 'Pricing Issue (Fix Margin)'
].sort_values('Total_Profit').head(5)

# Weak products to review or drop
top_drop = product_strategy[
    product_strategy['Product_Action'] == 'Weak Product (Review/Drop)'
].sort_values('Total_Profit').head(5)



In [33]:
print("\n=== CORE PRODUCTS (KEEP & PROMOTE) ===")
display(top_keep[['Product Name','Total_Profit','Distribution_Points']])


=== CORE PRODUCTS (KEEP & PROMOTE) ===


,Product Name,Total_Profit,Distribution_Points
10266,Canon imageCLASS 2200 Advanced Copier,25199.9280,3
3150,"Hoover Stove, Red",10345.5840,3
11636,"Nokia Smart Phone, Full Size",8121.4800,3
4417,Fellowes PB500 Electric Punch Plastic Comb Bin...,7753.0390,4
9953,Hewlett Packard LaserJet 3310 Copier,6983.8836,4


These products generate high profit and already have strong market presence across multiple regions.

Products like Canon imageCLASS Advanced Copier, Nokia Smart Phone – Full Size, and HP LaserJet Copier are consistently profitable and widely distributed.

👉 These are the company’s strongest performers and should be prioritized for inventory availability, promotions, and long-term product strategy.

Business takeaway:
Focus marketing and supply efforts on these products because they drive most of the profit.

In [34]:
print("\n=== GROWTH OPPORTUNITIES (EXPAND REACH) ===")
display(top_expand[['Product Name','Total_Profit','Distribution_Points']])


=== GROWTH OPPORTUNITIES (EXPAND REACH) ===


,Product Name,Total_Profit,Distribution_Points
842,"SAFCO Executive Leather Armchair, Black",5003.1000,2
560,"Safco Classic Bookcase, Metal",4681.8570,2
9789,"Canon Wireless Fax, Laser",4596.3240,2
10185,Canon PC1060 Personal Laser Copier,4570.9347,2
10517,HP Designjet T520 Inkjet Large Format Printer ...,4094.9766,2


These products are highly profitable but currently sold in fewer regions.

Items such as Safco Executive Armchair, Canon Wireless Fax, and HP Designjet Printer perform very well where they are available.

👉 This indicates strong demand but limited distribution.

Business takeaway:
Expanding these products into more regions could significantly increase total revenue and profit.

In [35]:
print("\n=== PRICING ISSUES (FIX MARGIN) ===")
display(top_fix_pricing[['Product Name','Total_Profit','Distribution_Points']])


=== PRICING ISSUES (FIX MARGIN) ===


,Product Name,Total_Profit,Distribution_Points
2844,"Hoover Stove, White",-4958.1630,3
11382,"Apple Smart Phone, Full Size",-4574.6439,3
2175,Chromcraft Bull-Nose Wood Oval Conference Tabl...,-2876.1156,4
7849,"Fellowes File Cart, Single Width",-2430.9180,3
10500,"Okidata Inkjet, Wireless",-2052.2340,3


These products are widely distributed but losing money.

For example, Hoover Stove – White and Apple Smart Phone – Full Size appear in many regions but generate negative profit.

👉 Customers are buying them, but heavy discounts, high costs, or poor pricing strategy are hurting margins.

Business takeaway:
These products should not be removed immediately — instead pricing, discounting, or cost structure should be reviewed to restore profitability.

In [36]:
print("\n=== WEAK PRODUCTS (REVIEW / DROP) ===")
display(top_drop[['Product Name','Total_Profit','Distribution_Points']])


=== WEAK PRODUCTS (REVIEW / DROP) ===


,Product Name,Total_Profit,Distribution_Points
10458,Cubify CubeX 3D Printer Double Head Print,-8879.9704,2
10495,Lexmark MX611dhe Monochrome Laser Printer,-4589.9730,2
10896,"Motorola Smart Phone, Cordless",-3998.6820,2
10782,Cubify CubeX 3D Printer Triple Head Print,-3839.9904,1
769,"Office Star Executive Leather Armchair, Black",-3066.7830,2


These products show low profit and limited distribution.

Products like Cubify 3D Printers and Lexmark Laser Printer are both under-distributed and heavily loss-making.

👉 There is weak demand and poor profitability.

Business takeaway:
These products are strong candidates for discontinuation, replacement, or major repositioning.